# Functionalities tutorial for FloatMatcher library


In [3]:
import warnings

import numpy as np
import xarray as xr

from floatmatcher.matchup import NearestNeighbor
from floatmatcher.orchestrator import Orchestrator
from floatmatcher.products import ERA5Product
from floatmatcher.profile_loader import ProfileLoader

### Create points to matchup
Cases : 
* from arrays (tryptic lon, lat, time)
* from dataframe with custom mapping
* from xarray dataset with custom mapping

Usage :
``` fm.data([data]*, mapping=[])``` 

features :
* good loader selection is triggered by the [data]* type



In [ ]:
# Random points case
rng = np.random.default_rng(0)
n_points = 50

lon = rng.uniform(-180.0, 180.0, n_points)
lat = rng.uniform(-60.0, 60.0, n_points)
day_offsets = rng.integers(0, 10, size=n_points).astype("timedelta64[D]")
hour_offsets = rng.integers(0, 24, size=n_points).astype("timedelta64[h]")
time = (np.datetime64("2018-01-01") + day_offsets + hour_offsets).astype("datetime64[ns]")


positions = np.array([lon, lat, time])
points_from_arrays = ProfileLoader.from_arrays(lon, lat, time)

# -------------------------------------------

# xarray dataset case
ds_profiles = xr.Dataset(
    {
        "LONGITUDE": (("N_PROF",), np.array([-40.0, -30.0, -20.0])),
        "LATITUDE": (("N_PROF",), np.array([35.0, 36.0, 37.0])),
        "TIME": (("N_PROF",), np.array(["2018-01-03", "2018-01-04", "2018-01-05"],
                                       dtype="datetime64[ns]")),
    },
    coords={"N_PROF": np.array([10, 11, 12])},
)

points_from_dataset = ProfileLoader.from_xrdataset(ds_profiles)

# res = orchestrator(points, vars, product).match(method=nearest, max_dist, max_time)

########## wanted : ##########
# instance floatmatcher : 
import floatmatcher as fm
my_fm = fm.new() 

# enter input parameters : 
# 1) match positions : lon lat time
#   1.1) custom series (3 arrays)
#   1.2) dataframe + mapping dictionnary
#   1.3) dataset xr + mapping dictionnary
#   1.4) argopy (no mapping needed) K
my_serie=...
my_fm.input_positions.from_arrays(my_serie)

# 2) matched product 
#   2.1) product type 
#   2.2) product source (local / remote)
#   2.3) Subset variables: all or list of output product variables (add enum available)

# 3) matching method
    # 3.1) method selection (Nearest / interpolation)
    # 3.2) method configuration 
    #   radius (km)
    #   time
    #

# get normalized positions : fm.load_positions()
# 

# ??? 

### data to matchup with points
Cases :
* from a file path
* from an explicit list of file paths
* from a repository
* from a tree and a fine pattern
* from remote 

Usage : 
``` fm.product()

In [8]:
# ---- load product with simple resolver : case of given list of files ----
file_list = ["/runtime/data/era5_daily/2018/01/era5_single-levels_20180101.nc",
             "/runtime/data/era5_daily/2018/01/era5_single-levels_20180102.nc",
             "/runtime/data/era5_daily/2018/01/era5_single-levels_20180103.nc",
             ]

product = ERA5Product.from_local(path=file_list)



# ---- load product with simple resolver : case of given list of files ----
product = ERA5Product.from_local(path="/runtime/data/era5_daily/2018/01")



# ---- load product with path + pattern ----
ERA5_ROOT = "/runtime/data/era5_daily"
ERA5_PATTERN = "{year}/{month:02d}/era5_single-levels_{year}{month:02d}{day:02d}.nc"

# more files than days because of the time pad taking into account files before and after
product = ERA5Product.from_local(path=ERA5_ROOT, pattern=ERA5_PATTERN)



########## wanted : ##########
# from floatmatcher.products import prFetcher 
# product = prFetcher(path, pattern, src=local, **kwargs (for local or remote src)) ?? 


### perform the matchup 
Set up the orchestrator. 
The orhestrator gather points, product and selected variables to permorm the match on. 


In [11]:
result = Orchestrator(points=array_points, variables=["sst"],
                              product=product).match(NearestNeighbor(max_dist_km=300,
                                                                    max_time=np.timedelta64(6, "h")))

########## wanted : ##########
# res = orchestrator(points, vars, product).match(method=nearest, max_dist, max_time)

In [13]:
kept = result.valid
print(f"   sst        : {np.round(result.values['sst'][kept][:5], 2)} ")
print(f"   distance   : {np.round(result.distance_km[kept][:5], 1)} km")
print(f"   time gap   : {np.round(result.time_delta[kept] /3600, 1)[:5]} h")


   sst        : [   nan 293.53 280.77 285.88 301.21] 
   distance   : [ 7.4 12.1  8.   4.6  9.8] km
   time gap   : [6. 5. 5. 3. 5.] h


In [ ]:
result = Orchestrator(points=array_points, variables=["sst"],
                              product=product).match(NearestNeighbor(max_dist_km=300,
                                                                    max_time=np.timedelta64(6, "h")))


enriched = argo_result.to_dataset()
print("\ndataset enriched with:", list(enriched.data_vars))
print(enriched["sst_coloc"].values)
print(enriched)
